In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, Polygon
from datetime import datetime, timedelta
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import glob


In [2]:

# Use a raw string (r"...") so Windows backslashes are not treated as escape characters
file_path = r"G:\ASCII-ICE\1132_026560_20050101_014337_20050101_021752_full.pkl"

# Load the DataFrame
df = pd.read_pickle(file_path)

# Inspect the loaded data
print("Shape:", df.shape)
display(df.head())


Shape: (501, 7)


,datetime,geoc_lat,geoc_long,local_time,altitude,spectrum0,spectrum1
0,2005-01-01 01:43:36.606,70.9423,152.5599,11.8975,711.242,"[2.3874683380126953, 2.3458333015441895, 2.214...","[2.4266841411590576, 2.3458333015441895, 2.097..."
1,2005-01-01 01:43:40.702,70.7172,152.2218,11.8761,711.251,"[2.0737428665161133, 1.7183823585510254, 1.352...","[2.2306056022644043, 2.1889705657958984, 2.018..."
2,2005-01-01 01:43:44.798,70.4921,151.8836,11.8547,711.260,"[1.6815860271453857, 1.5223039388656616, 1.352...","[2.8972723484039307, 2.581127405166626, 2.0580..."
3,2005-01-01 01:43:48.894,70.2670,151.5454,11.8333,711.269,"[2.7011938095092773, 2.5419118404388428, 2.097...","[2.5443310737609863, 2.3850491046905518, 2.175..."
4,2005-01-01 01:43:52.990,70.0418,151.2073,11.8119,711.278,"[1.7208017110824585, 1.6791666746139526, 1.352...","[1.3678605556488037, 1.2085784673690796, 1.038..."


In [7]:
file_path2 = r"G:\Demeter-LSTM_AE\Data\Bg_window_data\Background_data-window_1.pkl"
df2 = pd.read_pickle(file_path2)
print(df2.head()
)

                          Q3_fb_1      lat       lon  fit_Q3_fb_1  \
datetime                                                            
2005-04-01 02:43:38.981  0.475704  64.1968  130.2277     0.330513   
2005-04-01 02:45:41.861 -0.710571  57.0570  125.1966     0.330514   
2005-04-01 02:47:44.741 -0.867434  49.8082  121.6397    -1.015481   
2005-04-01 02:49:47.621 -0.847826  42.5016  118.8833    -0.996073   
2005-04-01 02:51:50.256 -1.092924  35.1763  116.6047    -0.996074   

                         Res_Q3_fb_1   Q3_fb_2  fit_Q3_fb_2  Res_Q3_fb_2  \
datetime                                                                   
2005-04-01 02:43:38.981     0.145190  0.076206    -0.301398     0.377604   
2005-04-01 02:45:41.861    -1.041085 -1.254880    -0.301390    -0.953490   
2005-04-01 02:47:44.741     0.148048 -1.408428    -1.057652    -0.350776   
2005-04-01 02:49:47.621     0.148247 -1.722153    -1.020384    -0.701769   
2005-04-01 02:51:50.256    -0.096850 -1.888207    -1.020375 

In [8]:
df2[:4]

,Q3_fb_1,lat,lon,fit_Q3_fb_1,Res_Q3_fb_1,Q3_fb_2,fit_Q3_fb_2,Res_Q3_fb_2,Q3_fb_3,fit_Q3_fb_3,...,Res_Q3_fb_8,Q3_fb_9,fit_Q3_fb_9,Res_Q3_fb_9,Q3_fb_10,fit_Q3_fb_10,Res_Q3_fb_10,Q3_fb_11,fit_Q3_fb_11,Res_Q3_fb_11
datetime,,,,,,,,,,,,,,,,,,,,,
2005-04-01 02:43:38.981,0.475704,64.1968,130.2277,0.330513,0.145190,0.076206,-0.301398,0.377604,0.587068,0.004656,...,-3.270242,-2.483157,0.307161,-2.790318,-2.003250,0.051201,-2.054451,-1.870909,-0.410051,-1.460858
2005-04-01 02:45:41.861,-0.710571,57.0570,125.1966,0.330514,-1.041085,-1.254880,-0.301390,-0.953490,-1.263379,0.004666,...,-3.295517,-2.531417,0.307173,-2.838590,-2.428780,0.051214,-2.479994,-2.426355,-0.410038,-2.016317
2005-04-01 02:47:44.741,-0.867434,49.8082,121.6397,-1.015481,0.148048,-1.408428,-1.057652,-0.350776,-1.310114,-0.716088,...,-1.148169,-2.174456,-1.427648,-0.746808,-2.165684,-1.688240,-0.477444,-2.011867,-2.059759,0.047891
2005-04-01 02:49:47.621,-0.847826,42.5016,118.8833,-0.996073,0.148247,-1.722153,-1.020384,-0.701769,-1.469432,-0.829320,...,-1.682700,-2.735447,-1.396383,-1.339063,-2.725534,-1.509474,-1.216060,-2.627537,-1.801599,-0.825938


In [4]:
import os
import glob
import re
import pandas as pd

# Directory containing the background window files
folder_path = r"G:\Demeter-LSTM_AE\Data\Bg_window_data"

# Find all matching window pickle files
file_paths = glob.glob(os.path.join(folder_path, "Background_data-window_*.pkl"))

# Helper function to sort files numerically by window number (0, 1, 2, ... 19)
def extract_window_number(file_path):
    match = re.search(r"window_(\d+)", os.path.basename(file_path))
    return int(match.group(1)) if match else -1

file_paths = sorted(file_paths, key=extract_window_number)

# Process each file to extract start and end datetimes
records = []
for file_path in file_paths:
    filename = os.path.basename(file_path)
    window_id = extract_window_number(file_path)
    
    # Load DataFrame
    df = pd.read_pickle(file_path)
    
    # Extract start and end datetimes (files use a DatetimeIndex)
    if isinstance(df.index, pd.DatetimeIndex):
        start_dt = df.index.min()
        end_dt = df.index.max()
    elif "datetime" in df.columns:
        dt_col = pd.to_datetime(df["datetime"])
        start_dt = dt_col.min()
        end_dt = dt_col.max()
    else:
        dt_idx = pd.to_datetime(df.index)
        start_dt = dt_idx.min()
        end_dt = dt_idx.max()
    
    duration = end_dt - start_dt
    
    records.append({
        "window": window_id,
        "filename": filename,
        "start_datetime": start_dt,
        "end_datetime": end_dt,
        "duration_days": duration.days,
        "total_rows": len(df)
    })

# Create the summary DataFrame
window_summary_df = pd.DataFrame(records)

# Display the summary table
display(window_summary_df)

# Optional: save the summary table to a CSV file
# window_summary_df.to_csv(os.path.join(folder_path, "window_datetime_summary.csv"), index=False)


,window,filename,start_datetime,end_datetime,duration_days,total_rows
0,0,Background_data-window_0.pkl,2005-01-01 00:05:36.579,2006-03-31 23:04:22.917,454,84467
1,1,Background_data-window_1.pkl,2005-04-01 02:43:38.981,2006-06-30 23:46:22.173,455,85122
2,2,Background_data-window_2.pkl,2005-07-01 02:26:08.945,2006-09-30 18:31:22.780,456,88423
3,3,Background_data-window_3.pkl,2005-10-20 08:16:30.172,2006-12-31 22:58:22.625,437,88644
4,4,Background_data-window_4.pkl,2006-01-01 09:19:06.195,2007-03-31 23:10:20.781,454,93666
5,5,Background_data-window_5.pkl,2006-04-01 00:08:36.863,2007-06-30 23:53:23.270,455,98086
6,6,Background_data-window_6.pkl,2006-07-01 00:50:36.120,2007-09-30 23:28:50.791,456,100667
7,7,Background_data-window_7.pkl,2006-10-01 18:29:06.102,2007-12-31 23:03:53.903,456,101349
8,8,Background_data-window_8.pkl,2007-01-01 00:02:36.571,2008-03-31 23:45:53.267,455,103517
9,9,Background_data-window_9.pkl,2007-04-01 00:14:37.640,2008-06-23 17:31:23.714,449,103092


In [9]:
eq = pd.read_csv("D:\GIT\Demeter-Anomaly-Detection-Framework\Data\EQ.csv")

In [10]:
eq

,Time,lat,long,depth,mag
0,2005-01-01T01:42:24.850Z,7.293,93.919,30.0,5.0
1,2005-01-01T01:43:40.140Z,7.034,92.462,35.5,5.2
2,2005-01-01T01:55:28.460Z,2.910,95.623,24.5,5.7
3,2005-01-01T04:03:10.990Z,5.465,94.398,36.0,5.7
4,2005-01-01T06:25:44.820Z,5.099,92.304,11.7,6.7
...,...,...,...,...,...
12437,2010-12-30T19:56:36.380Z,50.380,153.964,190.1,5.0
12438,2010-12-30T21:22:30.350Z,-19.984,168.353,20.2,5.0
12439,2010-12-30T23:47:03.930Z,-31.830,-178.135,35.0,5.0
12440,2010-12-31T04:11:03.180Z,-19.209,167.902,10.0,5.1
